# 01 — Data Collection & Dataset Validation

This notebook loads **only the datasets supplied for this project** and verifies their structure, size, columns, missing values, and date coverage.


## Step 1 — Import libraries and define paths

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

print('Project:', PROJECT_ROOT)
print('Raw data:', RAW_DIR)


Project: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1
Raw data: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\data\raw


## Step 2 — Load the three main datasets

In [4]:
supply_chain = pd.read_csv(RAW_DIR / 'supply_chain.csv')
marketing = pd.read_csv(RAW_DIR / 'marketing_campaign.csv')

# news_sentiment.csv has no header row in the supplied file.
news = pd.read_csv(
    RAW_DIR / 'news_sentiment.csv',
    header=None,
    names=['sentiment', 'text'],
    encoding='latin1'
)

print('Supply chain:', supply_chain.shape)
print('Marketing:', marketing.shape)
print('News:', news.shape)


Supply chain: (5000, 14)
Marketing: (200000, 16)
News: (4846, 2)


## Step 3 — Load the supplied historical trade datasets

In [5]:
HIST_DIR = RAW_DIR / 'historical_trade'

historical = {
    'commodity_prices': pd.read_csv(HIST_DIR / 'commodity_prices_supply_chain.csv'),
    'disruption_events': pd.read_csv(HIST_DIR / 'disruption_events.csv'),
    'industry_exposure': pd.read_csv(HIST_DIR / 'industry_exposure.csv'),
    'port_congestion': pd.read_csv(HIST_DIR / 'port_congestion.csv'),
    'shipping_rates': pd.read_csv(HIST_DIR / 'shipping_rates.csv'),
    'tariff_timeline': pd.read_csv(HIST_DIR / 'tariff_timeline.csv'),
    'trade_flows': pd.read_csv(HIST_DIR / 'trade_flows.csv')
}

for name, df in historical.items():
    print(f'{name:22s} -> {df.shape}')


commodity_prices       -> (110515, 8)
disruption_events      -> (58, 18)
industry_exposure      -> (250, 15)
port_congestion        -> (6260, 11)
shipping_rates         -> (300, 12)
tariff_timeline        -> (71, 17)
trade_flows            -> (1250, 11)


## Step 4 — Check the main dataset columns

In [6]:
print('SUPPLY CHAIN COLUMNS')
print(supply_chain.columns.tolist())

print('\nNEWS COLUMNS')
print(news.columns.tolist())

print('\nMARKETING COLUMNS')
print(marketing.columns.tolist())


SUPPLY CHAIN COLUMNS
['Shipment_ID', 'Date', 'Origin_Port', 'Destination_Port', 'Transport_Mode', 'Product_Category', 'Distance_km', 'Weight_MT', 'Fuel_Price_Index', 'Geopolitical_Risk_Score', 'Weather_Condition', 'Carrier_Reliability_Score', 'Lead_Time_Days', 'Disruption_Occurred']

NEWS COLUMNS
['sentiment', 'text']

MARKETING COLUMNS
['Campaign_ID', 'Company', 'Campaign_Type', 'Target_Audience', 'Duration', 'Channel_Used', 'Conversion_Rate', 'Acquisition_Cost', 'ROI', 'Location', 'Language', 'Clicks', 'Impressions', 'Engagement_Score', 'Customer_Segment', 'Date']


## Step 5 — Check target variable

In [7]:
target = 'Disruption_Occurred'

print('Target:', target)
print(supply_chain[target].value_counts(dropna=False).sort_index())
print('\nTarget distribution:')
print((supply_chain[target].value_counts(normalize=True).sort_index() * 100).round(2))


Target: Disruption_Occurred
Disruption_Occurred
0    1937
1    3063
Name: count, dtype: int64

Target distribution:
Disruption_Occurred
0    38.74
1    61.26
Name: proportion, dtype: float64


## Step 6 — Check dates

In [8]:
supply_chain['Date'] = pd.to_datetime(supply_chain['Date'], errors='coerce')
marketing['Date'] = pd.to_datetime(marketing['Date'], errors='coerce')

print('Supply-chain date range:', supply_chain['Date'].min(), 'to', supply_chain['Date'].max())
print('Marketing date range:', marketing['Date'].min(), 'to', marketing['Date'].max())
print('News has date column:', any(c.lower() in {'date', 'published_date', 'news_date'} for c in news.columns))


Supply-chain date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00
Marketing date range: 2021-01-01 00:00:00 to 2021-12-31 00:00:00
News has date column: False


## Step 7 — Check missing values and duplicates

This is **validation only**. Cleaning will be done in `02_data_cleaning.ipynb`.

In [9]:
print('Supply-chain missing values:')
display(supply_chain.isna().sum().sort_values(ascending=False).to_frame('missing'))

print('Supply-chain duplicate rows:', supply_chain.duplicated().sum())
print('Marketing duplicate rows:', marketing.duplicated().sum())
print('News duplicate rows:', news.duplicated().sum())


Supply-chain missing values:


,missing
Shipment_ID,0
Date,0
Origin_Port,0
Destination_Port,0
Transport_Mode,0
Product_Category,0
Distance_km,0
Weight_MT,0
Fuel_Price_Index,0
Geopolitical_Risk_Score,0


Supply-chain duplicate rows: 0
Marketing duplicate rows: 0
News duplicate rows: 6


## Step 8 — Create a dataset inventory

In [10]:
inventory = pd.DataFrame([
    ['supply_chain.csv', len(supply_chain), len(supply_chain.columns), 'Main shipment disruption prediction'],
    ['news_sentiment.csv', len(news), len(news.columns), 'NLP sentiment and event signals'],
    ['marketing_campaign.csv', len(marketing), len(marketing.columns), 'Marketing performance / decision layer'],
], columns=['dataset', 'rows', 'columns', 'project_role'])

display(inventory)


,dataset,rows,columns,project_role
0,supply_chain.csv,5000,14,Main shipment disruption prediction
1,news_sentiment.csv,4846,2,NLP sentiment and event signals
2,marketing_campaign.csv,200000,16,Marketing performance / decision layer


## Step 9 — Important research validation

The supplied news dataset contains `sentiment` and `text`, but **no date**. Therefore we will not invent dates or falsely join news records to shipment dates.

The supplied marketing data is also from a different time period than the main shipment dataset, so it will be treated as a marketing/decision layer unless a valid common key is established.

**Next notebook:** `02_data_cleaning.ipynb`.